In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("OPENAI_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"OPENAI_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("OPENAI_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'OPENAI_API_KEY', and load_dotenv() ran without error.")

OPENAI_API_KEY loaded (164 characters): sk-p...VmEA


In [2]:
load_dotenv(override=True)

True

In [ ]:
"""
Entity type resolution, STEP 1: cluster only.

Embeds unique entities currently typed 'other' and clusters them.
Outputs a spreadsheet you can review and hand-edit before any LLM
call happens, move an entity to a different cluster_id, split a
cluster by giving some rows a new cluster_id, or merge two clusters
by giving them the same cluster_id.

Once you're satisfied with the groupings, feed the edited file into
step2_label_type_clusters.py.

Requires OPENAI_API_KEY as an environment variable (embeddings only,
no LLM calls in this step).
Designed for Jupyter/Colab execution. No __main__ guard.
"""

import os
import time
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = "C:\\Users\\olagunju\\OneDrive\\KSU PROJECT\\SHOLA_KSU_PUBLISHED_PAPERS\\Data-Minning\\NER-PROJECT\\PART-4-Entity-Resolution\\final_resolved_triples.xlsx"  # your current best-resolved entity set
SOURCE_COL = "final_source"
TARGET_COL = "final_target"
SOURCE_TYPE_COL = "source_type"
TARGET_TYPE_COL = "target_type"
DOI_COL = "doi"
OTHER_LABEL = "other"

EMBED_MODEL = "text-embedding-3-small"
DISTANCE_THRESHOLD = 0.25   # lower = stricter/smaller clusters, tune and rerun as needed

OUTPUT_CLUSTERS_XLSX = "entity_type_clusters_for_review.xlsx"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ---------------------------------------------------------------
# LOAD AND ISOLATE 'other'-TYPED ENTITIES
# ---------------------------------------------------------------
df = pd.read_excel(TRIPLES_PATH)

source_other = df.loc[df[SOURCE_TYPE_COL].astype(str).str.strip().str.lower() == OTHER_LABEL, [SOURCE_COL, DOI_COL]].rename(columns={SOURCE_COL: "entity"})
target_other = df.loc[df[TARGET_TYPE_COL].astype(str).str.strip().str.lower() == OTHER_LABEL, [TARGET_COL, DOI_COL]].rename(columns={TARGET_COL: "entity"})
other_long = pd.concat([source_other, target_other])
other_long["entity"] = other_long["entity"].astype(str).str.strip()

other_entities = sorted(other_long["entity"].unique())

# per-entity doi info, used later to show whether a cluster's members
# come from the same paper(s) or genuinely span the corpus
entity_doi_counts = other_long.groupby("entity")[DOI_COL].nunique().to_dict()
entity_dois = other_long.groupby("entity")[DOI_COL].apply(lambda s: sorted(s.unique())).to_dict()

print(f"Unique entities currently typed 'other': {len(other_entities)}")

# ---------------------------------------------------------------
# EMBED
# ---------------------------------------------------------------
def embed_batch(texts, batch_size=200):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        vectors.extend([d.embedding for d in resp.data])
        time.sleep(0.2)
    return np.array(vectors)


embeddings = embed_batch(other_entities)
print(f"Embedded {embeddings.shape[0]} entities into {embeddings.shape[1]}-dim vectors")

# ---------------------------------------------------------------
# CLUSTER
# ---------------------------------------------------------------
clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=DISTANCE_THRESHOLD,
    metric="cosine",
    linkage="average",
)
cluster_ids = clustering.fit_predict(embeddings)

cluster_df = pd.DataFrame({"entity": other_entities, "cluster_id": cluster_ids})
cluster_df["doc_frequency"] = cluster_df["entity"].map(entity_doi_counts)
cluster_df["dois"] = cluster_df["entity"].map(lambda e: "; ".join(entity_dois[e]))
cluster_df = cluster_df.sort_values(["cluster_id", "entity"]).reset_index(drop=True)

# cluster-level check: does this cluster's members overlap in DOI, or are
# they independent mentions across different papers? A cluster whose
# members share the SAME single paper is weaker synonym evidence than
# one where each member appears independently across many different papers.
# Built directly from the entity_dois dict (sets of actual DOI strings),
# not from the formatted "dois" display column, to avoid any join/split
# separator mismatch.
cluster_doi_union = cluster_df.groupby("cluster_id")["entity"].apply(
    lambda members: set().union(*[set(entity_dois[e]) for e in members])
)
cluster_df["cluster_unique_doi_count"] = cluster_df["cluster_id"].map(cluster_doi_union.apply(len))

cluster_sizes = cluster_df["cluster_id"].value_counts()
print(f"Formed {cluster_df['cluster_id'].nunique()} clusters")
print(f"  Singletons: {(cluster_sizes == 1).sum()}")
print(f"  Clusters with 2+ members: {(cluster_sizes >= 2).sum()}")
print(f"  Largest cluster size: {cluster_sizes.max()}")

# ---------------------------------------------------------------
# SAVE FOR MANUAL REVIEW
# ---------------------------------------------------------------
cluster_df["notes"] = ""  # blank column for you to jot down observations while reviewing

readme_rows = [
    "HOW TO READ THIS FILE",
    "",
    "- entity: an entity currently typed 'other' in source_type/target_type.",
    "",
    "- cluster_id: the group this entity was automatically placed in. Edit freely: change a",
    "  cluster_id to move an entity, give two rows the same cluster_id to merge groups,",
    "  give a row a new unused cluster_id to split it out on its own.",
    "",
    "- doc_frequency: how many unique papers mention this specific entity.",
    "",
    "- cluster_unique_doi_count: total distinct papers spanned by the WHOLE cluster combined,",
    "  a rough confidence signal, a cluster backed by many independent papers is more likely",
    "  a genuine, corpus-wide category than one paper's idiosyncratic phrasing.",
    "",
    "- notes: blank, for your own review comments.",
    "",
    "Once reviewed, run step2_label_type_clusters.py on this file.",
]
readme_df = pd.DataFrame({"": readme_rows})

with pd.ExcelWriter(OUTPUT_CLUSTERS_XLSX) as writer:
    readme_df.to_excel(writer, sheet_name="READ_ME_FIRST", index=False)
    cluster_df.to_excel(writer, sheet_name="clusters_for_review", index=False)

print(f"\nSaved clusters to {OUTPUT_CLUSTERS_XLSX}")
print("Review and edit cluster_id values as needed (merge, split, move entities).")
print("Then run step2_label_type_clusters.py on this file.")

Unique entities currently typed 'other': 1418
Embedded 1418 entities into 1536-dim vectors
Formed 974 clusters
  Singletons: 713
  Clusters with 2+ members: 261
  Largest cluster size: 10

Saved clusters to entity_type_clusters_for_review.xlsx
Review and edit cluster_id values as needed (merge, split, move entities).
Then run step2_label_type_clusters.py on this file.
